# Definining Cohort

**The goal of this notebook is to identify patients with advanced head and neck cancer who received first-line treatment with pembrolizumab plus platinum-based chemotherapy or pembrolizumab.**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## 1. Pembrolizumab plus platinum-based chemotherapy

In [3]:
therapy = pd.read_csv('../data/LineOfTherapy.csv')

In [4]:
therapy_1l = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
)

In [5]:
(
    therapy_1l[
    therapy_1l['LineName'].str.contains('Pembrolizumab')
    & therapy_1l['LineName'].str.contains('Carboplatin|Cisplatin')
    & ~therapy_1l['LineName'].str.contains('Clinical Study Drug')]
    .LineName.value_counts()
)

LineName
Carboplatin,Fluorouracil,Pembrolizumab                           345
Carboplatin,Paclitaxel,Pembrolizumab                             172
Cisplatin,Fluorouracil,Pembrolizumab                              67
Carboplatin,Pembrolizumab                                         26
Carboplatin,Docetaxel,Pembrolizumab                               19
Cisplatin,Pembrolizumab                                           11
Capecitabine,Carboplatin,Pembrolizumab                             8
Carboplatin,Paclitaxel Protein-Bound,Pembrolizumab                 7
Carboplatin,Fluorouracil,Paclitaxel,Pembrolizumab                  6
Cisplatin,Paclitaxel,Pembrolizumab                                 5
Carboplatin,Cetuximab,Fluorouracil,Pembrolizumab                   4
Cisplatin,Docetaxel,Fluorouracil,Pembrolizumab                     4
Cisplatin,Docetaxel,Pembrolizumab                                  3
Carboplatin,Docetaxel,Fluorouracil,Pembrolizumab                   3
Cisplatin,Fluorouracil,Le

In [6]:
pembro_plat_df = (
    therapy_1l[
    therapy_1l['LineName'].str.contains('Pembrolizumab')
    & therapy_1l['LineName'].str.contains('Carboplatin|Cisplatin')
    & ~therapy_1l['LineName'].str.contains('Clinical Study Drug')]
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'pembro_platinum'))

In [7]:
pembro_plat_df.sample(3)

,PatientID,LineName,StartDate
6297,F20D0B0D470E6,pembro_platinum,2022-12-30
1665,FE41819E32C0D,pembro_platinum,2021-12-06
5539,F8D2F6E3EF770,pembro_platinum,2020-05-22


In [8]:
row_ID(pembro_plat_df)

(705, 705)

## 2. Pembrolizumab 

In [9]:
pembro_df = (
    therapy_1l
    .query('LineName == "Pembrolizumab"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'pembro'))

In [10]:
pembro_df.sample(3)

,PatientID,LineName,StartDate
5789,F539FE3B3299F,pembro,2017-11-28
2871,FE52A02455A50,pembro,2020-11-02
5498,FF189EEAD84D2,pembro,2021-05-03


In [11]:
row_ID(pembro_df)

(1149, 1149)

## 3. Combine dataframes and export to csv 

In [12]:
firstline_pembrochemo_pembro_index = pd.concat([pembro_plat_df, pembro_df], axis = 0)

In [13]:
row_ID(firstline_pembrochemo_pembro_index)

(1854, 1854)

In [14]:
firstline_pembrochemo_pembro_index.sample(3)

,PatientID,LineName,StartDate
17047,FCA51859B73B4,pembro_platinum,2023-02-02
10962,F4B65508B1D57,pembro,2022-03-04
15793,F2D94F2380999,pembro,2022-01-03


In [15]:
firstline_pembrochemo_pembro_index.LineName.value_counts()

LineName
pembro             1149
pembro_platinum     705
Name: count, dtype: int64

In [16]:
firstline_pembrochemo_pembro_index.to_csv('../outputs/pembrochemo_pembro_index.csv', index = False)